In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm.auto import tqdm

tqdm.pandas()

In [144]:
DATASET_PATH = '../dataset/E066/'

In [ ]:
def encode_histone_exp(row, histone):
    arr = []
    start = row['start']
    end = row['end']
    gene_id = row.index[0]
    count = 1

    for i in range(start, end, 100):
        histone_df = histone.loc[
                    # The same gene ID
                    (histone.index == gene_id) &
                    (
                        # The window is inside the histone
                        ((histone['chromStart'] <= i) & (histone['chromEnd'] >= i + 100)) |
                        # The start of histone is inside the window
                        ((histone['chromStart'] >= i) & (histone['chromStart'] <= i + 100) & (histone['chromEnd'] >= i + 100)) |
                        # The end of histone is inside the window
                        ((histone['chromStart'] <= i) & (histone['chromEnd'] >= i) & (histone['chromEnd'] <= i + 100))
                    )
                    ]
        size = histone_df.shape[0]
        if (size > 0):
            avg = histone_df['signalValue'].mean()
        else:
            avg = 0.0
        
        # print(f"Bin {count}: {i} to {i + 100}: {histone_df.shape[0]}, average: {avg}")
        arr.append(avg)
        count += 1
    return arr

In [ ]:
# Load the gene expression file
print("Loading the gene expression file")

column_names = ['chromosome_name',
                'start',
                'end',
                'gene_id',
                'E066',
                'strand',
                'label',
                'external_gene_name',
                'start_position',
                'end_position',
                'tss' 
                ]

E066_df = pd.read_csv(os.path.join(DATASET_PATH, "E066.bed"), sep="\t", names = column_names)
print(f"E066 file: {E066_df.shape}")

In [ ]:
E066_df

In [ ]:
E066_df.set_index('gene_id', inplace=True)

In [ ]:
E066_df

In [ ]:
E066_df.info()

In [ ]:
# Load the histone data
print("Loading the histone file")
H3K4me1_df = pd.read_csv(os.path.join(DATASET_PATH, "E066_H3K4me1_df.csv"))
H3K4me3_df = pd.read_csv(os.path.join(DATASET_PATH, "E066_H3K4me3_df.csv"))
H3K9me3_df = pd.read_csv(os.path.join(DATASET_PATH, "E066_H3K9me3_df.csv"))
H3K27me3_df = pd.read_csv(os.path.join(DATASET_PATH, "E066_H3K27me3_df.csv"))
H3K36me3_df = pd.read_csv(os.path.join(DATASET_PATH, "E066_H3K36me3_df.csv"))

print(H3K4me1_df.shape)
print(H3K4me3_df.shape)
print(H3K9me3_df.shape)
print(H3K27me3_df.shape)
print(H3K36me3_df.shape)

In [ ]:
H3K4me1_df

In [ ]:
# Set the index
H3K4me1_df.set_index('gene_id', inplace=True)
H3K4me3_df.set_index('gene_id', inplace=True)
H3K9me3_df.set_index('gene_id', inplace=True)
H3K27me3_df.set_index('gene_id', inplace=True)
H3K36me3_df.set_index('gene_id', inplace=True)

In [ ]:
print("Generate histone column")
E066_df.loc[:, 'H3K4me1'] = E066_df.progress_apply(lambda row: encode_histone_exp(row, H3K4me1_df), axis = 1)

# Try with polars

In [4]:
import polars as pl
import os

In [5]:
# Load the gene expression file
schema = pl.Schema({
        'chromosome_name': pl.String,
        'start': pl.Int64,
        'end': pl.Int64,
        'gene_id': pl.String,
        'E066': pl.Float64,
        'strand': pl.Int64,
        'label': pl.Int64,
        'external_gene_name': pl.String,
        'start_position': pl.Int64,
        'end_position': pl.Int64,
        'tss': pl.Int64
})

In [7]:
DATASET_PATH = '../dataset/E066/'

In [8]:
E066_pl = pl.read_csv(os.path.join(DATASET_PATH, "E066.bed"), 
                      separator="\t", 
                      schema=schema,
                      has_header=False,
                      skip_rows=0)

In [9]:
E066_pl

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988
"""chrX""",99834799,99844799,"""ENSG00000000005""",0.191,1,-1,"""TNMD""",99839799,99854882,99839799
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092
"""chr1""",169858408,169868408,"""ENSG00000000457""",4.733,-1,1,"""SCYL3""",169818772,169863408,169863408
"""chr1""",169626245,169636245,"""ENSG00000000460""",0.942,1,-1,"""C1orf112""",169631245,169823221,169631245
…,…,…,…,…,…,…,…,…,…,…
"""chr15""",102280913,102290913,"""ENSG00000259658""",0.212,-1,-1,"""RP11-89K11.1""",102277302,102285913,102285913
"""chr15""",97966182,97976182,"""ENSG00000259664""",0.0,-1,-1,"""CTD-2147F2.2""",97913601,97971182,97971182
"""chr16""",33642696,33652696,"""ENSG00000259680""",0.071,-1,-1,"""RP11-812E19.9""",33647044,33647696,33647696


In [10]:
print("Loading the histone file")
H3K4me1_pl = pl.read_csv(os.path.join(DATASET_PATH, "E066_H3K4me1_df.csv"))

Loading the histone file


In [11]:
H3K4me1_pl

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,chrom,chromStart,chromEnd,name,score,strand_peak,signalValue,pValue,qValue,peak,startBucket,endBucket
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64,str,i64,i64,str,i64,str,f64,f64,f64,i64,f64,f64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49575664,49577113,"""Rank_14""",734,""".""",20.49891,73.43851,67.02312,620,55.72,70.21
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49572952,49574573,"""Rank_188""",564,""".""",18.24062,56.41204,51.06926,350,28.6,44.81
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49571998,49572915,"""Rank_71909""",102,""".""",5.67759,10.27033,8.2097,169,19.06,28.23
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49577186,49577379,"""Rank_159548""",51,""".""",3.58302,5.16452,3.52579,35,70.94,72.87
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr14""",103584344,103594344,"""ENSG00000259717""",0.0,-1,-1,"""LINC00677""",103587184,103589344,103589344,"""chr14""",103585293,103585664,"""Rank_133666""",61,""".""",4.02373,6.15866,4.43337,135,9.49,13.2
"""chr14""",103584344,103594344,"""ENSG00000259717""",0.0,-1,-1,"""LINC00677""",103587184,103589344,103589344,"""chr14""",103584522,103585231,"""Rank_139790""",58,""".""",3.18223,5.88999,4.18566,143,1.78,8.87
"""chr14""",103584344,103594344,"""ENSG00000259717""",0.0,-1,-1,"""LINC00677""",103587184,103589344,103589344,"""chr14""",103591686,103592058,"""Rank_145378""",55,""".""",3.67747,5.55802,3.87557,247,73.42,77.14


In [80]:
genes_sample= E066_pl.filter(pl.col("gene_id").is_in(["ENSG00000000419", "ENSG00000000003"]))

In [81]:
genes_sample

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092


In [82]:
H3K4me1_pl.filter(pl.col("gene_id").is_in(["ENSG00000000419", "ENSG00000000003"]))

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,chrom,chromStart,chromEnd,name,score,strand_peak,signalValue,pValue,qValue,peak,startBucket,endBucket
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64,str,i64,i64,str,i64,str,f64,f64,f64,i64,f64,f64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49575664,49577113,"""Rank_14""",734,""".""",20.49891,73.43851,67.02312,620,55.72,70.21
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49572952,49574573,"""Rank_188""",564,""".""",18.24062,56.41204,51.06926,350,28.6,44.81
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49571998,49572915,"""Rank_71909""",102,""".""",5.67759,10.27033,8.2097,169,19.06,28.23
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49577186,49577379,"""Rank_159548""",51,""".""",3.58302,5.16452,3.52579,35,70.94,72.87
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49571244,49571632,"""Rank_161879""",51,""".""",3.68107,5.10401,3.47041,307,11.52,15.4
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49577629,49577806,"""Rank_163065""",50,""".""",3.52959,5.07326,3.44426,40,75.37,77.14


In [113]:
# genes = genes_sample
genes = E066_pl
histone = H3K4me1_pl

In [114]:
 # Create a sequence of window starts for each gene
genes_with_windows = genes.with_columns([
    pl.int_ranges(pl.col('start'), pl.col('end'), 100).alias('window_starts')
]).explode('window_starts')

In [115]:
genes_with_windows = genes_with_windows.with_columns([
        (pl.col('window_starts') + 100).alias('window_ends')
    ])

In [116]:
genes_with_windows

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,window_starts,window_ends
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64,i64,i64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99889988,99890088
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890088,99890188
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890188,99890288
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890288,99890388
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890388,99890488
…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr15""",47421320,47431320,"""ENSG00000259752""",0.0,-1,-1,"""FKSG62""",47425693,47426320,47426320,47430820,47430920
"""chr15""",47421320,47431320,"""ENSG00000259752""",0.0,-1,-1,"""FKSG62""",47425693,47426320,47426320,47430920,47431020
"""chr15""",47421320,47431320,"""ENSG00000259752""",0.0,-1,-1,"""FKSG62""",47425693,47426320,47426320,47431020,47431120


In [117]:
# Join genes with histone data
joined = genes_with_windows.join(
    histone,
    left_on='gene_id',
    right_on='gene_id',
    how='left'
)

In [118]:
joined

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,window_starts,window_ends,chromosome_name_right,start_right,end_right,E066_right,strand_right,label_right,external_gene_name_right,start_position_right,end_position_right,tss_right,chrom,chromStart,chromEnd,name,score,strand_peak,signalValue,pValue,qValue,peak,startBucket,endBucket
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64,i64,i64,str,i64,i64,f64,i64,i64,str,i64,i64,i64,str,i64,i64,str,i64,str,f64,f64,f64,i64,f64,f64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99889988,99890088,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890088,99890188,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890188,99890288,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890288,99890388,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890388,99890488,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr15""",47421320,47431320,"""ENSG00000259752""",0.0,-1,-1,"""FKSG62""",47425693,47426320,47426320,47430820,47430920,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr15""",47421320,47431320,"""ENSG00000259752""",0.0,-1,-1,"""FKSG62""",47425693,47426320,47426320,47430920,47431020,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr15""",47421320,47431320,"""ENSG00000259752""",0.0,-1,-1,"""FKSG62""",47425693,47426320,47426320,47431020,47431120,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [119]:
# Fill missing values with 0.0
joined = joined.with_columns([
    pl.col('signalValue').fill_null(0.0),
    pl.col('chromStart').fill_null(pl.col('window_starts')),
    pl.col('chromEnd').fill_null(pl.col('window_ends'))
])

In [120]:
joined

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,window_starts,window_ends,chromosome_name_right,start_right,end_right,E066_right,strand_right,label_right,external_gene_name_right,start_position_right,end_position_right,tss_right,chrom,chromStart,chromEnd,name,score,strand_peak,signalValue,pValue,qValue,peak,startBucket,endBucket
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64,i64,i64,str,i64,i64,f64,i64,i64,str,i64,i64,i64,str,i64,i64,str,i64,str,f64,f64,f64,i64,f64,f64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99889988,99890088,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890088,99890188,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890188,99890288,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890288,99890388,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890388,99890488,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr15""",47421320,47431320,"""ENSG00000259752""",0.0,-1,-1,"""FKSG62""",47425693,47426320,47426320,47430820,47430920,null,null,null,null,null,null,null,null,null,null,null,47430820,47430920,null,null,null,0.0,null,null,null,null,null
"""chr15""",47421320,47431320,"""ENSG00000259752""",0.0,-1,-1,"""FKSG62""",47425693,47426320,47426320,47430920,47431020,null,null,null,null,null,null,null,null,null,null,null,47430920,47431020,null,null,null,0.0,null,null,null,null,null
"""chr15""",47421320,47431320,"""ENSG00000259752""",0.0,-1,-1,"""FKSG62""",47425693,47426320,47426320,47431020,47431120,null,null,null,null,null,null,null,null,null,null,null,47431020,47431120,null,null,null,0.0,null,null,null,null,null


In [121]:
# Filter and calculate average signal value
result = joined.filter(
    (pl.col('chromStart') <= pl.col('window_starts')) & 
    (pl.col('chromEnd') >= pl.col('window_ends')) |
    (pl.col('chromStart') >= pl.col('window_starts')) & 
    (pl.col('chromStart') <= pl.col('window_ends')) & 
    (pl.col('chromEnd') >= pl.col('window_ends')) |
    (pl.col('chromStart') <= pl.col('window_starts')) & 
    (pl.col('chromEnd') >= pl.col('window_starts')) & 
    (pl.col('chromEnd') <= pl.col('window_ends'))
)

result

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,window_starts,window_ends,chromosome_name_right,start_right,end_right,E066_right,strand_right,label_right,external_gene_name_right,start_position_right,end_position_right,tss_right,chrom,chromStart,chromEnd,name,score,strand_peak,signalValue,pValue,qValue,peak,startBucket,endBucket
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64,i64,i64,str,i64,i64,f64,i64,i64,str,i64,i64,i64,str,i64,i64,str,i64,str,f64,f64,f64,i64,f64,f64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890188,99890288,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890288,99890388,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890388,99890488,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890488,99890588,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890588,99890688,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr15""",47421320,47431320,"""ENSG00000259752""",0.0,-1,-1,"""FKSG62""",47425693,47426320,47426320,47430820,47430920,null,null,null,null,null,null,null,null,null,null,null,47430820,47430920,null,null,null,0.0,null,null,null,null,null
"""chr15""",47421320,47431320,"""ENSG00000259752""",0.0,-1,-1,"""FKSG62""",47425693,47426320,47426320,47430920,47431020,null,null,null,null,null,null,null,null,null,null,null,47430920,47431020,null,null,null,0.0,null,null,null,null,null
"""chr15""",47421320,47431320,"""ENSG00000259752""",0.0,-1,-1,"""FKSG62""",47425693,47426320,47426320,47431020,47431120,null,null,null,null,null,null,null,null,null,null,null,47431020,47431120,null,null,null,0.0,null,null,null,null,null


In [122]:
gene_wo_histone = genes_with_windows.filter(~pl.col('window_starts').is_in(result['window_starts']))

In [123]:
gene_wo_histone.shape

(1091193, 13)

In [124]:
gene_wo_histone = gene_wo_histone.with_columns(
    chromosome_name_right = pl.lit(None).cast(pl.String),
    start_right = pl.lit(0).cast(pl.Int64),
    end_right =  pl.lit(0).cast(pl.Int64),
    E066_right = pl.lit(0.0).cast(pl.Float64),
    strand_right = pl.lit(0).cast(pl.Int64),
    label_right = pl.lit(0).cast(pl.Int64),
    external_gene_name_right = pl.lit(None).cast(pl.String),
    start_position_right = pl.lit(0).cast(pl.Int64),
    end_position_right = pl.lit(0).cast(pl.Int64),
    tss_right = pl.lit(0).cast(pl.Int64),
    chrom = pl.lit(None).cast(pl.String),
    chromStart = pl.lit(0).cast(pl.Int64),
    chromEnd = pl.lit(0).cast(pl.Int64),
    name = pl.lit(None).cast(pl.String),
    score = pl.lit(0).cast(pl.Int64),
    strand_peak = pl.lit(None).cast(pl.String),
    signalValue = pl.lit(0.0).cast(pl.Float64),
    pValue = pl.lit(0.0).cast(pl.Float64),
    qValue = pl.lit(0.0).cast(pl.Float64),
    peak = pl.lit(0).cast(pl.Int64),
    startBucket = pl.lit(0.0).cast(pl.Float64),
    endBucket = pl.lit(0.0).cast(pl.Float64)
)

In [125]:
gene_wo_histone

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,window_starts,window_ends,chromosome_name_right,start_right,end_right,E066_right,strand_right,label_right,external_gene_name_right,start_position_right,end_position_right,tss_right,chrom,chromStart,chromEnd,name,score,strand_peak,signalValue,pValue,qValue,peak,startBucket,endBucket
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64,i64,i64,str,i64,i64,f64,i64,i64,str,i64,i64,i64,str,i64,i64,str,i64,str,f64,f64,f64,i64,f64,f64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99889988,99890088,null,0,0,0.0,0,0,null,0,0,0,null,0,0,null,0,null,0.0,0.0,0.0,0,0.0,0.0
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890088,99890188,null,0,0,0.0,0,0,null,0,0,0,null,0,0,null,0,null,0.0,0.0,0.0,0,0.0,0.0
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99891088,99891188,null,0,0,0.0,0,0,null,0,0,0,null,0,0,null,0,null,0.0,0.0,0.0,0,0.0,0.0
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99891188,99891288,null,0,0,0.0,0,0,null,0,0,0,null,0,0,null,0,null,0.0,0.0,0.0,0,0.0,0.0
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99891288,99891388,null,0,0,0.0,0,0,null,0,0,0,null,0,0,null,0,null,0.0,0.0,0.0,0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr14""",103584344,103594344,"""ENSG00000259717""",0.0,-1,-1,"""LINC00677""",103587184,103589344,103589344,103591444,103591544,null,0,0,0.0,0,0,null,0,0,0,null,0,0,null,0,null,0.0,0.0,0.0,0,0.0,0.0
"""chr14""",103584344,103594344,"""ENSG00000259717""",0.0,-1,-1,"""LINC00677""",103587184,103589344,103589344,103591544,103591644,null,0,0,0.0,0,0,null,0,0,0,null,0,0,null,0,null,0.0,0.0,0.0,0,0.0,0.0
"""chr14""",103584344,103594344,"""ENSG00000259717""",0.0,-1,-1,"""LINC00677""",103587184,103589344,103589344,103593444,103593544,null,0,0,0.0,0,0,null,0,0,0,null,0,0,null,0,null,0.0,0.0,0.0,0,0.0,0.0


In [126]:
gene_wo_histone = gene_wo_histone.group_by(['gene_id', 'window_starts']).agg([
    pl.col('signalValue').mean().alias('avg_signal')
]).sort(['gene_id', 'window_starts'])

In [127]:
gene_wo_histone

gene_id,window_starts,avg_signal
str,i64,f64
"""ENSG00000000003""",99889988,0.0
"""ENSG00000000003""",99890088,0.0
"""ENSG00000000003""",99891088,0.0
"""ENSG00000000003""",99891188,0.0
"""ENSG00000000003""",99891288,0.0
…,…,…
"""ENSG00000259717""",103591444,0.0
"""ENSG00000259717""",103591544,0.0
"""ENSG00000259717""",103593444,0.0


In [128]:
# Filter and calculate average signal value
result = joined.filter(
    (pl.col('chromStart') <= pl.col('window_starts')) & 
    (pl.col('chromEnd') >= pl.col('window_ends')) |
    (pl.col('chromStart') >= pl.col('window_starts')) & 
    (pl.col('chromStart') <= pl.col('window_ends')) & 
    (pl.col('chromEnd') >= pl.col('window_ends')) |
    (pl.col('chromStart') <= pl.col('window_starts')) & 
    (pl.col('chromEnd') >= pl.col('window_starts')) & 
    (pl.col('chromEnd') <= pl.col('window_ends'))
).group_by(['gene_id', 'window_starts']).agg([
    pl.col('signalValue').mean().alias('avg_signal')
]).sort(['gene_id', 'window_starts'])

In [129]:
result

gene_id,window_starts,avg_signal
str,i64,f64
"""ENSG00000000003""",99890188,5.92171
"""ENSG00000000003""",99890288,5.92171
"""ENSG00000000003""",99890388,5.92171
"""ENSG00000000003""",99890488,5.92171
"""ENSG00000000003""",99890588,5.92171
…,…,…
"""ENSG00000259752""",47430820,0.0
"""ENSG00000259752""",47430920,0.0
"""ENSG00000259752""",47431020,0.0


In [130]:
result.extend(gene_wo_histone)

gene_id,window_starts,avg_signal
str,i64,f64
"""ENSG00000000003""",99890188,5.92171
"""ENSG00000000003""",99890288,5.92171
"""ENSG00000000003""",99890388,5.92171
"""ENSG00000000003""",99890488,5.92171
"""ENSG00000000003""",99890588,5.92171
…,…,…
"""ENSG00000259717""",103591444,0.0
"""ENSG00000259717""",103591544,0.0
"""ENSG00000259717""",103593444,0.0


In [131]:
result = result.sort("window_starts")

In [132]:
result

gene_id,window_starts,avg_signal
str,i64,f64
"""ENSG00000253896""",17601,0.0
"""ENSG00000253896""",17701,0.0
"""ENSG00000253896""",17801,0.0
"""ENSG00000253896""",17901,0.0
"""ENSG00000253896""",18001,0.0
…,…,…
"""ENSG00000185220""",249204895,0.0
"""ENSG00000185220""",249204995,0.0
"""ENSG00000185220""",249205095,0.0


In [140]:
final_result = result.group_by('gene_id').agg(
    pl.col('avg_signal')
).sort('gene_id')

In [142]:
final_result

gene_id,avg_signal
str,list[f64]
"""ENSG00000000003""","[0.0, 0.0, … 0.0]"
"""ENSG00000000005""","[0.0, 0.0, … 0.0]"
"""ENSG00000000419""","[0.0, 0.0, … 0.0]"
"""ENSG00000000457""","[0.0, 0.0, … 0.0]"
"""ENSG00000000460""","[0.0, 0.0, … 0.0]"
…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]"
"""ENSG00000259664""","[0.0, 0.0, … 0.0]"
"""ENSG00000259680""","[0.0, 0.0, … 0.0]"


In [162]:
print(final_result.filter(pl.col('gene_id').is_in(["ENSG00000000003"]))[0, 1].to_list())

[0.0, 0.0, 5.92171, 5.92171, 5.92171, 5.92171, 5.92171, 5.92171, 5.92171, 5.92171, 5.92171, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [155]:
final_result

gene_id,avg_signal
str,list[f64]
"""ENSG00000000003""","[0.0, 0.0, … 0.0]"
"""ENSG00000000005""","[0.0, 0.0, … 0.0]"
"""ENSG00000000419""","[0.0, 0.0, … 0.0]"
"""ENSG00000000457""","[0.0, 0.0, … 0.0]"
"""ENSG00000000460""","[0.0, 0.0, … 0.0]"
…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]"
"""ENSG00000259664""","[0.0, 0.0, … 0.0]"
"""ENSG00000259680""","[0.0, 0.0, … 0.0]"


In [161]:
final_result.with_columns(
    pl.col("avg_signal").list.len().alias('length')
)

gene_id,avg_signal,length
str,list[f64],u32
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",100
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",100
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",100
…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",100
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",100
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",100


In [148]:
final_result.write_parquet(os.path.join(DATASET_PATH, 'E066_H3K4me1.parquet'))

# Check the Parquet File

This data was from previous step

In [149]:
loaded_pl = pl.read_parquet(os.path.join(DATASET_PATH, 'E066_H3K4me1.parquet'))

In [150]:
loaded_pl

gene_id,avg_signal
str,list[f64]
"""ENSG00000000003""","[0.0, 0.0, … 0.0]"
"""ENSG00000000005""","[0.0, 0.0, … 0.0]"
"""ENSG00000000419""","[0.0, 0.0, … 0.0]"
"""ENSG00000000457""","[0.0, 0.0, … 0.0]"
"""ENSG00000000460""","[0.0, 0.0, … 0.0]"
…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]"
"""ENSG00000259664""","[0.0, 0.0, … 0.0]"
"""ENSG00000259680""","[0.0, 0.0, … 0.0]"
